In [ ]:
import asyncio
import inspect
import json
import os
import re
import threading
import uuid
from abc import ABC, abstractmethod
from datetime import datetime, timedelta
from enum import Enum
from pathlib import Path
from typing import Annotated, Callable, List, get_args, get_origin, get_type_hints
from zoneinfo import ZoneInfo

import anthropic
import numpy as np
from anthropic.types import UserMessageParam, AssistantMessageParam, ToolParam, TextBlockParam, ToolUseBlockParam, ToolResultBlockParam
from mcp import Client
from openai import OpenAI
from openai.types.chat import ChatCompletionSystemMessageParam, ChatCompletionUserMessageParam, ChatCompletionAssistantMessageParam
from openai.types.chat import ChatCompletionToolParam, ChatCompletionMessageToolCallParam, ChatCompletionToolMessageParam

In [ ]:
def note(label: str, text: str):
    """Print one step of the agent's work, indented, so it stands apart from questions and answers."""
    lines = str(text).splitlines() or [""]
    print(f"  * {label}: {lines[0]}")
    for line in lines[1:]:
        print(f"    {line}")

In [ ]:
class MsgType(Enum):
    SYSTEM = "System"
    USER = "User"
    ASSISTANT = "Assistant"
    TOOL = "Tool"

In [ ]:
class Message(ABC):

    def __init__(self, msg_type: MsgType):
        self.msg_type = msg_type

    @abstractmethod
    def summary(self) -> str:
        pass

    def to_dict(self) -> dict:
        return {"type": self.msg_type.value, "content": self.summary()}

    def __repr__(self) -> str:
        return f"Message({self.msg_type.value}): {self.summary()}"

In [ ]:
class SystemMessage(Message):

    def __init__(self, description: str):
        super().__init__(MsgType.SYSTEM)
        self.description = description

    def summary(self) -> str:
        return self.description

In [ ]:
class UserMessage(Message):

    def __init__(self, request: str):
        super().__init__(MsgType.USER)
        self.request = request

    def summary(self) -> str:
        return self.request

In [ ]:
class ToolCall:

    def __init__(self, id: str, name: str, arguments: dict):
        self.id = id
        self.name = name
        self.arguments = arguments

    def to_dict(self) -> dict:
        return {"id": self.id, "name": self.name, "arguments": self.arguments}

    def __repr__(self) -> str:
        return f"{self.name}({json.dumps(self.arguments, ensure_ascii=False)})"

In [ ]:
class AssistantMessage(Message):

    def __init__(self, content: str, tool_calls: List[ToolCall] = None, thinking: List[dict] = None):
        super().__init__(MsgType.ASSISTANT)
        self.content = content
        self.tool_calls = tool_calls or []
        self.thinking = thinking or []  # the model's reasoning blocks, passed back to the API unchanged

    def thinking_text(self) -> str:
        return "\n".join(block.get("thinking", "") for block in self.thinking).strip()

    def summary(self) -> str:
        if self.tool_calls:
            return f"{self.content} [call: {self.tool_calls}]"
        return self.content

    def to_dict(self) -> dict:
        return {"type": self.msg_type.value, "content": self.content,
                "tool_calls": [call.to_dict() for call in self.tool_calls], "thinking": self.thinking}

In [ ]:
class ToolResultMessage(Message):

    def __init__(self, tool_call_id: str, name: str, result: str):
        super().__init__(MsgType.TOOL)
        self.tool_call_id = tool_call_id
        self.name = name
        self.result = result

    def summary(self) -> str:
        return f"{self.name} -> {self.result}"

    def to_dict(self) -> dict:
        return {"type": self.msg_type.value, "tool_call_id": self.tool_call_id,
                "name": self.name, "content": self.result}

In [ ]:
def message_from_dict(data: dict) -> Message:
    if data["type"] == MsgType.ASSISTANT.value:
        return AssistantMessage(data["content"], [ToolCall(**c) for c in data.get("tool_calls", [])], data.get("thinking", []))
    if data["type"] == MsgType.TOOL.value:
        return ToolResultMessage(data["tool_call_id"], data["name"], data["content"])
    classes = {MsgType.SYSTEM.value: SystemMessage, MsgType.USER.value: UserMessage}
    return classes[data["type"]](data["content"])

In [ ]:
system_message = SystemMessage("You're a learning assistant for the book 《从大模型到智能体》 (From LLM to Agent). "
                               "You help beginners learn how to build AI agents.")

In [ ]:
class Tool:

    def __init__(self, name: str, description: str, parameters: dict, function: Callable):
        self.name = name
        self.description = description
        self.parameters = parameters  # JSON Schema of the arguments
        self.function = function

    def run(self, arguments: dict) -> str:
        return str(self.function(**arguments))

    def __call__(self, *args, **kwargs):
        return self.function(*args, **kwargs)

In [ ]:
JSON_TYPES = {str: "string", int: "integer", float: "number", bool: "boolean", list: "array", dict: "object"}


def tool(function: Callable) -> Tool:
    """Decorator: build a Tool from a function's name, docstring and type hints."""
    hints = get_type_hints(function, include_extras=True)
    properties, required = {}, []

    for name, param in inspect.signature(function).parameters.items():
        hint, description = hints.get(name, str), None
        if get_origin(hint) is Annotated:  # Annotated[int, "description"]
            hint, description = get_args(hint)[:2]
        properties[name] = {"type": JSON_TYPES.get(hint, "string")}
        if description:
            properties[name]["description"] = description
        if param.default is inspect.Parameter.empty:
            required.append(name)

    return Tool(
        name=function.__name__,
        description=inspect.getdoc(function) or "",
        parameters={"type": "object", "properties": properties, "required": required},
        function=function
    )

In [ ]:
@tool
def get_current_time(timezone: Annotated[str, "IANA timezone name, e.g. 'America/Toronto', 'Asia/Shanghai'"]) -> str:
    """Get the current local date and time in a given IANA timezone."""
    return datetime.now(ZoneInfo(timezone)).strftime("%Y-%m-%d %H:%M:%S %Z")


print(json.dumps(get_current_time.parameters, indent=2))
print(get_current_time("America/Toronto"))

In [ ]:
@tool
def add_days(date: Annotated[str, "Start date in YYYY-MM-DD format"],
             days: Annotated[int, "Number of days to add, negative to subtract"] = 1) -> str:
    """Add (or subtract, if negative) a number of days to a date. Returns the new date and its weekday."""
    result = datetime.strptime(date, "%Y-%m-%d") + timedelta(days=days)
    return result.strftime("%Y-%m-%d (%A)")


print(json.dumps(add_days.parameters, indent=2))
print(add_days("2026-11-10", 100))

In [ ]:
class ToolRegistry:

    def __init__(self, tools: List[Tool] = None):
        self.tools = {}
        for t in tools or []:
            self.register(t)

    def register(self, t: Tool) -> Tool:
        if t.name in self.tools:
            raise ValueError(f"duplicate tool name: '{t.name}'")
        self.tools[t.name] = t
        return t

    def list(self) -> List[Tool]:
        return list(self.tools.values())

    def run(self, call: ToolCall) -> str:
        t = self.tools.get(call.name)
        if t is None:
            return f"Error: unknown tool '{call.name}'"
        try:
            return t.run(call.arguments)
        except Exception as e:
            return f"Error: {e}"  # let the model see the error and try to fix it

In [ ]:
class MCPConnection:
    """A synchronous wrapper around the async mcp.Client (Part 6 will make the whole Agent async).

    The connection lives in one long-running coroutine on a background event loop;
    every call is submitted to that loop and we wait for its result.
    """

    def __init__(self, url: str):  # the URL of a remote MCP server
        self.loop = asyncio.new_event_loop()
        threading.Thread(target=self.loop.run_forever, daemon=True).start()
        ready = threading.Event()
        self.session = asyncio.run_coroutine_threadsafe(self._session(url, ready), self.loop)
        ready.wait()
        if self.session.done():  # connecting failed: raise the error here
            self.session.result()

    async def _session(self, url: str, ready: threading.Event):
        self.closed = asyncio.Event()
        try:
            async with Client(url) as client:
                self.client = client
                ready.set()
                await self.closed.wait()  # keep the connection open until close()
        finally:
            ready.set()

    def _run(self, coroutine):
        return asyncio.run_coroutine_threadsafe(coroutine, self.loop).result()

    def list_tools(self):
        return self._run(self.client.list_tools()).tools

    def call_tool(self, name: str, arguments: dict):
        return self._run(self.client.call_tool(name, arguments))

    def close(self):
        self.loop.call_soon_threadsafe(self.closed.set)
        self.session.result()
        self.loop.call_soon_threadsafe(self.loop.stop)

In [ ]:
class MCPTool(Tool):
    """A tool that lives on an MCP server: same interface as Tool, but run() goes over MCP."""

    def __init__(self, connection: MCPConnection, mcp_tool):
        super().__init__(mcp_tool.name, mcp_tool.description or "", mcp_tool.input_schema, function=None)
        self.connection = connection

    def run(self, arguments: dict) -> str:
        result = self.connection.call_tool(self.name, arguments)
        text = "\n".join(block.text for block in result.content if block.type == "text")
        if result.is_error and not text.startswith("Error"):
            text = f"Error: {text}"  # same convention as ToolRegistry.run()
        return text


def load_mcp_tools(connection: MCPConnection) -> List[Tool]:
    return [MCPTool(connection, t) for t in connection.list_tools()]

In [ ]:
class Memory:
    SUMMARIZE_SYSTEM_MSG = "You are a conversation summary assistant."
    SUMMARIZE_USER_MSG = "Please condense the following conversation into a brief summary: "
    SUMMARY_PREFIX = "Historical Summary: "

    def __init__(self, provider, max_tokens=1000, max_messages=3, keep_messages=1, memory_id=None):
        self.provider = provider
        self.id = memory_id or uuid.uuid4().hex[:8]
        self.messages: List[Message] = []
        self.max_tokens = max_tokens
        self.max_messages = max_messages
        self.keep_messages = keep_messages

    def add(self, *messages: Message):
        # add a whole turn at once, so it is summarized (at most) once per turn
        self.messages.extend(messages)
        if len(self.messages) > self.max_messages or self.token_count() > self.max_tokens:
            self.summarize()

    def reset(self):
        self.messages = []

    def history(self, messages: List[Message] = None) -> str:
        return "\n".join(
            f"[{msg.msg_type.value}]: {msg.summary()}" for msg in (messages or self.messages)
        )

    def summarize(self):
        if self.keep_messages >= len(self.messages):
            return

        split = len(self.messages) - self.keep_messages
        # never separate a tool call from its result: keep from the last UserMessage onwards
        while split > 0 and not isinstance(self.messages[split], UserMessage):
            split -= 1
        if split == 0:
            return
        to_summarize = self.messages[:split]
        to_keep = self.messages[split:]

        request = UserMessage(Memory.SUMMARIZE_USER_MSG + "\n" + self.history(to_summarize))
        summary = self.provider.call(SystemMessage(Memory.SUMMARIZE_SYSTEM_MSG), [request])
        self.messages = [UserMessage(Memory.SUMMARY_PREFIX + summary.summary())] + to_keep
        note("summary", self.history())

    def token_count(self):
        # rough estimate: counts characters, not real tokens
        return sum(len(msg.summary() or "") for msg in self.messages)

    def to_dict(self) -> dict:
        return {"id": self.id, "messages": [msg.to_dict() for msg in self.messages]}

    @classmethod
    def from_dict(cls, data: dict, provider) -> "Memory":
        memory = cls(provider, memory_id=data["id"])
        memory.messages = [message_from_dict(m) for m in data["messages"]]
        return memory

In [ ]:
class MemoryStore:

    def __init__(self, root="memories"):
        self.root = Path(root)
        self.root.mkdir(exist_ok=True)

    def save(self, memory: Memory):
        path = self.root / f"{memory.id}.json"
        path.write_text(json.dumps(memory.to_dict(), ensure_ascii=False, indent=2), encoding="utf-8")

    def load(self, memory_id: str, provider) -> Memory:
        path = self.root / f"{memory_id}.json"
        if not path.exists():
            return None
        return Memory.from_dict(json.loads(path.read_text(encoding="utf-8")), provider)

In [ ]:
class LLMProvider(ABC):

    @abstractmethod
    def call(self, system: SystemMessage, messages: List[Message], tools: List[Tool] = None, model=None, max_tokens=4096,
             output_schema: dict = None):
        pass

In [ ]:
class OpenAIProvider(LLMProvider):

    def __init__(self, effort: str = None):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=None
        )
        self.default_model = "gpt-5.4-mini"
        self.effort = effort  # "none", "low", "medium", "high" or "xhigh"

    def call(self, system: SystemMessage, messages: List[Message], tools: List[Tool] = None, model=None, max_tokens=4096,
             output_schema: dict = None):
        if system:
            messages = [system] + messages
        kwargs = dict(
            model=model or self.default_model,
            max_completion_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        if tools:
            kwargs["tools"] = self._tool_schema(tools)
        if self.effort:
            kwargs["reasoning_effort"] = self.effort  # the model thinks, but only the answer comes back
        if output_schema:  # the answer must be JSON that follows the schema
            kwargs["response_format"] = {"type": "json_schema",
                                         "json_schema": {"name": "output", "strict": True, "schema": output_schema}}
        response = self.client.chat.completions.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _tool_schema(tools: List[Tool]):
        return [
            ChatCompletionToolParam(type="function", function={
                "name": tool.name, "description": tool.description, "parameters": tool.parameters})
            for tool in tools
        ]

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                schema.append(ChatCompletionSystemMessageParam(role="system", content=msg.description))
            elif isinstance(msg, UserMessage):
                schema.append(ChatCompletionUserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                param = ChatCompletionAssistantMessageParam(role="assistant", content=msg.content)
                if msg.tool_calls:
                    param["tool_calls"] = [
                        ChatCompletionMessageToolCallParam(id=call.id, type="function", function={
                            "name": call.name, "arguments": json.dumps(call.arguments)})
                        for call in msg.tool_calls
                    ]
                schema.append(param)
            elif isinstance(msg, ToolResultMessage):
                # OpenAI: one "tool" message per tool result
                schema.append(ChatCompletionToolMessageParam(role="tool", tool_call_id=msg.tool_call_id, content=msg.result))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        message = response.choices[0].message
        tool_calls = [
            ToolCall(call.id, call.function.name, json.loads(call.function.arguments))
            for call in (message.tool_calls or [])
        ]
        return AssistantMessage(message.content or "", tool_calls)

In [ ]:
class ClaudeProvider(LLMProvider):

    def __init__(self, effort: str = None):
        self.client = anthropic.Anthropic(
            api_key=os.getenv("ANTHROPIC_API_KEY")
        )
        self.default_model = "claude-haiku-5-5"  # a reasoning model: thinking is on by default
        self.effort = effort  # "low", "medium", "high" (default), "xhigh" or "max"

    def call(self, system: SystemMessage, messages: List[Message], tools: List[Tool] = None, model=None, max_tokens=16000,
             output_schema: dict = None):
        kwargs = dict(
            model=model or self.default_model,
            max_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        if system:
            kwargs["system"] = system.summary()
        if tools:
            kwargs["tools"] = self._tool_schema(tools)
        kwargs["thinking"] = {"type": "adaptive", "display": "summarized"}  # return a summary of the thinking
        if self.effort:
            kwargs["output_config"] = {"effort": self.effort}
        if output_schema:  # the answer must be JSON that follows the schema
            kwargs.setdefault("output_config", {})["format"] = {"type": "json_schema", "schema": output_schema}
        response = self.client.messages.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _tool_schema(tools: List[Tool]):
        return [
            ToolParam(name=tool.name, description=tool.description, input_schema=tool.parameters)
            for tool in tools
        ]

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, UserMessage):
                schema.append(UserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                content = list(msg.thinking)  # thinking blocks first, unchanged: required during tool use
                content += [TextBlockParam(type="text", text=msg.content)] if msg.content else []
                content += [
                    ToolUseBlockParam(type="tool_use", id=call.id, name=call.name, input=call.arguments)
                    for call in msg.tool_calls
                ]
                schema.append(AssistantMessageParam(role="assistant", content=content))
            elif isinstance(msg, ToolResultMessage):
                # Claude: all tool results of one turn go into a single "user" message
                block = ToolResultBlockParam(type="tool_result", tool_use_id=msg.tool_call_id, content=msg.result)
                if schema and schema[-1]["role"] == "user" and isinstance(schema[-1]["content"], list):
                    schema[-1]["content"].append(block)
                else:
                    schema.append(UserMessageParam(role="user", content=[block]))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        text = "".join(block.text for block in response.content if block.type == "text")
        tool_calls = [
            ToolCall(block.id, block.name, block.input)
            for block in response.content if block.type == "tool_use"
        ]
        thinking = [
            {"type": "thinking", "thinking": block.thinking, "signature": block.signature} if block.type == "thinking"
            else {"type": "redacted_thinking", "data": block.data}
            for block in response.content if block.type in ("thinking", "redacted_thinking")
        ]
        return AssistantMessage(text, tool_calls, thinking)

In [ ]:
class EmbeddingProvider(ABC):
    model: str = ""  # name of the embedding model, saved together with an index

    @abstractmethod
    def embed(self, texts: List[str]) -> np.ndarray:
        """Return one vector per text, shape (len(texts), dimension)."""
        pass

In [ ]:
class OpenAIEmbedding(EmbeddingProvider):

    def __init__(self, model: str = "text-embedding-3-small"):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=None
        )
        self.model = model

    def embed(self, texts: List[str]) -> np.ndarray:
        response = self.client.embeddings.create(model=self.model, input=texts)
        data = sorted(response.data, key=lambda d: d.index)  # one embedding per input, in order
        return np.array([d.embedding for d in data])

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Cosine similarity between every row of a and every row of b, shape (len(a), len(b))."""
    a = a / np.linalg.norm(a, axis=1, keepdims=True)
    b = b / np.linalg.norm(b, axis=1, keepdims=True)
    return a @ b.T

In [ ]:
class Chunk:

    def __init__(self, text: str, source: str):
        self.text = text
        self.source = source  # which document it comes from, used for citations in chapter 15

    def to_dict(self) -> dict:
        return {"text": self.text, "source": self.source}


def load_documents(folder: str) -> List[tuple]:
    """Return (source, text) for every Markdown file in the folder."""
    return [(path.name, path.read_text(encoding="utf-8")) for path in sorted(Path(folder).glob("*.md"))]

In [ ]:
def chunk_fixed(text: str, size: int = 200, overlap: int = 50) -> List[str]:
    """Cut the text every `size` characters; neighbouring chunks share `overlap` characters."""
    step = size - overlap
    return [text[start:start + size] for start in range(0, max(len(text) - overlap, 1), step)]

In [ ]:
def chunk_by_heading(text: str, source: str, max_size: int = 500) -> List[Chunk]:
    """One chunk per "## " section, prefixed with its headings; long sections are cut again."""
    chunks, title, heading, lines = [], "", "", []

    def flush():
        body = "\n".join(lines).strip()
        if body:
            prefix = " > ".join(h for h in (title, heading) if h)
            pieces = chunk_fixed(body, max_size, overlap=50) if len(body) > max_size else [body]
            chunks.extend(Chunk(f"{prefix}\n{piece}", source) for piece in pieces)
        lines.clear()

    for line in text.splitlines():
        if line.startswith("# "):
            flush()
            title, heading = line[2:].strip(), ""
        elif line.startswith("## "):
            flush()
            heading = line[3:].strip()
        else:
            lines.append(line)
    flush()
    return chunks

In [ ]:
class VectorIndex:

    def __init__(self, embedder: EmbeddingProvider):
        self.embedder = embedder
        self.chunks: List[Chunk] = []
        self.vectors = None  # shape (len(chunks), dimension)

    def add(self, chunks: List[Chunk], batch_size: int = 100):
        batches = [chunks[i:i + batch_size] for i in range(0, len(chunks), batch_size)]
        vectors = np.vstack([self.embedder.embed([c.text for c in batch]) for batch in batches])
        self.vectors = vectors if self.vectors is None else np.vstack([self.vectors, vectors])
        self.chunks += chunks

    def search(self, query: str, top_k: int = 3, min_score: float = 0.0) -> List[tuple]:
        if not self.chunks:
            return []
        scores = cosine_similarity(self.embedder.embed([query]), self.vectors)[0]
        best = np.argsort(scores)[::-1][:top_k]
        return [(self.chunks[i], float(scores[i])) for i in best if scores[i] >= min_score]

    def save(self, folder: str):
        folder = Path(folder)
        folder.mkdir(exist_ok=True)
        np.save(folder / "vectors.npy", self.vectors)
        data = {"model": self.embedder.model, "chunks": [c.to_dict() for c in self.chunks]}
        (folder / "chunks.json").write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

    @classmethod
    def load(cls, folder: str, embedder: EmbeddingProvider) -> "VectorIndex":
        folder = Path(folder)
        data = json.loads((folder / "chunks.json").read_text(encoding="utf-8"))
        if data["model"] != embedder.model:  # vectors of different models can not be compared
            raise ValueError(f"index was built with '{data['model']}', not '{embedder.model}'")
        index = cls(embedder)
        index.chunks = [Chunk(**c) for c in data["chunks"]]
        index.vectors = np.load(folder / "vectors.npy")
        return index

In [ ]:
class LongTermMemory:
    EXTRACT_PROMPT = """From the conversation below, extract facts about the user that are worth remembering
in future conversations, such as their name, background, preferences and goals.
Write one short fact per line, in the same language as the user.
Do not repeat facts that are already known. If there is nothing new worth remembering, answer NONE.

Already known:
{known}

User: {user}
Assistant: {assistant}"""

    def __init__(self, provider: LLMProvider, embedder: EmbeddingProvider, folder: str = "long_term_memory"):
        self.provider = provider
        self.folder = folder
        if Path(folder, "chunks.json").exists():
            self.index = VectorIndex.load(folder, embedder)  # memories from earlier sessions
        else:
            self.index = VectorIndex(embedder)

    def recall(self, query: str, top_k: int = 3, min_score: float = 0.25) -> List[str]:
        return [chunk.text for chunk, _ in self.index.search(query, top_k, min_score)]

    def remember(self, user: UserMessage, assistant: AssistantMessage) -> List[str]:
        known = self.recall(user.request, top_k=5)
        prompt = LongTermMemory.EXTRACT_PROMPT.format(
            known="\n".join(known) or "(nothing)", user=user.request, assistant=assistant.content)
        response = self.provider.call(SystemMessage("You extract long-term memories about the user."), [UserMessage(prompt)])

        facts = [line.strip("-* ").strip() for line in response.content.splitlines()]
        facts = [fact for fact in facts if fact and fact.upper() != "NONE"]
        if facts:
            today = datetime.now().strftime("%Y-%m-%d")
            self.index.add([Chunk(fact, source=today) for fact in facts])  # source: when it was remembered
            self.index.save(self.folder)
        return facts

    def facts(self) -> List[str]:
        return [chunk.text for chunk in self.index.chunks]

In [ ]:
class Agent:

    def __init__(self, provider: LLMProvider, system: SystemMessage, registry: ToolRegistry = None, max_steps: int = 5,
                 long_term: LongTermMemory = None):
        self.provider = provider
        self.system = system
        self.registry = registry or ToolRegistry()
        self.max_steps = max_steps
        self.long_term = long_term
        self.store = MemoryStore()
        self.memory = Memory(provider)

    @staticmethod
    def show_thinking(response: AssistantMessage):
        if response.thinking_text():
            note("thinking", response.thinking_text())

    def run(self, message: str, system: SystemMessage = None) -> AssistantMessage:
        system = system or self.system
        memories = self.long_term.recall(message) if self.long_term else []
        if memories:  # put what we remember about the user into the system prompt
            for memory in memories:
                note("recall", memory)
            system = SystemMessage(system.summary() + "\n\nWhat you remember about the user:\n"
                                   + "\n".join(f"- {memory}" for memory in memories))
        tools = self.registry.list()
        turn = [UserMessage(message)]

        response = self.provider.call(system, self.memory.messages + turn, tools)
        self.show_thinking(response)
        turn.append(response)

        step = 0
        while response.tool_calls and step < self.max_steps:
            step += 1
            for call in response.tool_calls:
                result = self.registry.run(call)
                note(f"tool[{step}]", f"{call} -> {result}")
                turn.append(ToolResultMessage(call.id, call.name, result))
            response = self.provider.call(system, self.memory.messages + turn, tools)
            self.show_thinking(response)
            turn.append(response)

        if response.tool_calls:
            response.tool_calls = []
            response.content = response.content or f"(stopped after {self.max_steps} steps)"

        self.memory.add(*turn)
        self.store.save(self.memory)

        if self.long_term:  # extract new long-term facts from this turn
            for fact in self.long_term.remember(turn[0], response):
                note("remember", fact)
        return response

    def chat(self, questions: List[str] = None, system: SystemMessage = None):
        questions = iter(questions) if questions is not None else None  # preset questions, or None to type
        note("memory", f"#{self.memory.id}")
        while True:
            message = next(questions, "/quit") if questions else input("YOU: ")
            print(f"\nYOU: {message}")
            if message.strip().lower() == "/quit":
                break
            if not message.strip():
                continue

            if message.strip().lower() == "/new":
                self.memory = Memory(self.provider)
                note("new memory", f"#{self.memory.id}")
                continue

            if message.startswith("#"):
                memory = self.store.load(message[1:].strip(), self.provider)
                if memory is None:
                    note("memory", f"{message} not found")
                else:
                    self.memory = memory
                    note(f"loaded memory #{memory.id}", memory.history())
                continue

            response = self.run(message, system)
            print(f"AI: {response.content}")

In [ ]:
provider = OpenAIProvider()

EVENT_SCHEMA = {
    "type": "object",
    "properties": {
        "title": {"type": "string"},
        "date": {"type": "string", "description": "YYYY-MM-DD"},
        "people": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["title", "date", "people"],  # strict mode: every field is required
    "additionalProperties": False,
}

text = "2026 年 11 月 18 日下午，我和小王、小李在会议室讨论第 5 部分的大纲。"
response = provider.call(SystemMessage("Extract the event from the text."), [UserMessage(text)], output_schema=EVENT_SCHEMA)
print(response.content)

event = json.loads(response.content)  # always valid JSON, no parsing surprises
print(event["date"], event["people"])

In [ ]:
PLAN_SCHEMA = {
    "type": "object",
    "properties": {"steps": {"type": "array", "items": {"type": "string"}}},
    "required": ["steps"],
    "additionalProperties": False,
}

PLAN_PROMPT = """You are a planner. Break the user's task into a short list of concrete steps (at most 5).
Each step must be doable with these tools: {tools}
The last step combines the results into the final answer."""


def format_done(done: List[tuple]) -> str:
    return "\n".join(f"- {step} -> {result}" for step, result in done)


def make_plan(task: str, provider: LLMProvider, registry: ToolRegistry, done: List[tuple] = None) -> List[str]:
    prompt = PLAN_PROMPT.format(tools=", ".join(t.name for t in registry.list()))
    if done:  # replanning: keep what is done, plan only the rest
        prompt += f"\n\nThese steps are done, with their results:\n{format_done(done)}\nPlan only the remaining steps."
    response = provider.call(SystemMessage(prompt), [UserMessage(task)], output_schema=PLAN_SCHEMA)
    return json.loads(response.content)["steps"]


registry = ToolRegistry([get_current_time, add_days])
task = "我想从今天开始学习本书第 3 到第 5 部分，每部分学 4 天，学完一部分紧接着学下一部分。请列出每部分的开始和结束日期，以及星期几。"
for step in make_plan(task, provider, registry):
    print("-", step)

In [ ]:
STEP_PROMPT = """Overall task: {task}

Steps done so far, with their results:
{done}

Now do only this step: {step}
If this step can not be done, reply with "FAILED:" followed by the reason."""


def plan_and_execute(task: str, agent: Agent, max_replans: int = 1) -> str:
    steps = make_plan(task, agent.provider, agent.registry)
    print("PLAN:", *steps, sep="\n  - ")
    done, replans = [], 0

    while steps:
        step = steps.pop(0)
        print(f"\n>>> STEP: {step}")
        result = agent.run(STEP_PROMPT.format(task=task, done=format_done(done) or "(nothing yet)", step=step)).content
        print("RESULT:", result)
        done.append((step, result))

        if result.startswith("FAILED") and replans < max_replans:  # the plan did not work: make a new one
            replans += 1
            steps = make_plan(task, agent.provider, agent.registry, done)
            print("\nREPLAN:", *steps, sep="\n  - ")

    return done[-1][1]  # the last step gives the final answer

In [ ]:
agent = Agent(provider, system_message, registry=registry)
agent.memory = Memory(provider, max_messages=50, max_tokens=20000)  # each step prompt already carries the results
answer = plan_and_execute(task, agent)
print("\nFINAL ANSWER:\n", answer)

In [ ]:
# a task with a step our tools can not do: watch it fail and replan
answer = plan_and_execute("告诉我今天是几号，并在我的 Google 日历里加一个复习第 3 部分的提醒。", agent)
print("\nFINAL ANSWER:\n", answer)